In [0]:
import os
import re
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

# ====================================================
# Step 0: Dynamic file path (latest uploaded file)
# ====================================================
base_path = "dbfs:/FileStore/TreadingSample/"
latest_file = max([f.path for f in dbutils.fs.ls(base_path)], key=lambda x: x)
file_path = latest_file
filename = os.path.basename(file_path).replace(".csv", "")
delta_table_name = f"{filename}_delta"

# Drop table if exists
spark.sql(f"DROP TABLE IF EXISTS {delta_table_name}")

# ====================================================
# Step 1: Read CSV with first row as header
# ====================================================
df_temp = spark.read.csv(file_path, header=False, inferSchema=False)
header_row = df_temp.first()
df = df_temp.toDF(*[str(c) for c in header_row]).filter(F.col(header_row[0]) != header_row[0])

# Clean column names
def clean_column_name(c):
    return re.sub(r"[^0-9a-zA-Z_]", "_", c.strip())
df = df.toDF(*[clean_column_name(c) for c in df.columns])

# ====================================================
# Step 2: Detect column types using sample
# ====================================================
sample_df = df.limit(500)
cast_exprs = []

for col_name in sample_df.columns:
    sample_vals = sample_df.select(col_name).na.drop()

    # ISO 8601 Timestamp
    if sample_vals.filter(F.col(col_name).rlike(r"^\d{4}[-/]\d{2}[-/]\d{2}T")).count() > 0:
        cast_exprs.append(F.to_timestamp(F.col(col_name), "yyyy/MM/dd'T'HH:mm:ss.SSS'Z'").alias(col_name))
        continue

    # Date formats
    if sample_vals.filter(F.col(col_name).rlike(r"^\d{4}-\d{2}-\d{2}$|^\d{2}-\d{2}-\d{4}$")).count() > 0:
        cast_exprs.append(F.to_date(F.col(col_name), "yyyy-MM-dd").alias(col_name))
        continue

    # Integer
    if sample_vals.filter(~F.col(col_name).rlike(r"^-?\d+$")).count() == 0:
        cast_exprs.append(F.col(col_name).cast(IntegerType()).alias(col_name))
        continue

    # Decimal
    if sample_vals.filter(~F.col(col_name).rlike(r"^-?\d+(\.\d+)?$")).count() == 0:
        cast_exprs.append(F.col(col_name).cast(DoubleType()).alias(col_name))
        continue

    # Default: String
    cast_exprs.append(F.col(col_name).cast(StringType()).alias(col_name))

df_casted = df.select(*cast_exprs)

# ====================================================
# Step 3: Save as Delta
# ====================================================
df_casted.write.format("delta").mode("overwrite").saveAsTable(delta_table_name)
print(f"Delta table '{delta_table_name}' created successfully with correct data types.")


In [0]:
%sql
 select * from default.positionmanager_05_08_2025_delta

In [0]:
import os
from pyspark.sql import functions as F

# Read Delta table
df = spark.table(delta_table_name)

schema_info = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]
agg_exprs = []

# Build one-pass aggregation
for col_name, dtype in schema_info:
    if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
        agg_exprs.extend([
            F.min(F.col(col_name)).cast("string").alias(f"{col_name}_min"),
            F.max(F.col(col_name)).cast("string").alias(f"{col_name}_max"),
            F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls")
        ])
    else:
        agg_exprs.append(F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls"))

# Run aggregation
stats_row = df.agg(*agg_exprs).collect()[0]

# Convert to final tidy table
rows = []
for col_name, dtype in schema_info:
    if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
        rows.append((
            stats_row[f"{col_name}_min"],
            stats_row[f"{col_name}_max"],
            stats_row[f"{col_name}_nulls"],
            col_name,
            dtype,
            filename + ".csv"
        ))
    else:
        rows.append((
            None,
            None,
            stats_row[f"{col_name}_nulls"],
            col_name,
            dtype,
            filename + ".csv"
        ))

final_stats_df = spark.createDataFrame(rows, ["min_value", "max_value", "null_count", "column_name", "data_type", "filename"])
display(final_stats_df)


In [0]:
import os
import re
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

# ====================================================
# Step 0: Latest file path
# ====================================================
base_path = "dbfs:/FileStore/TreadingSample/"
latest_file_info = max(dbutils.fs.ls(base_path), key=lambda x: x.modificationTime)
file_path = latest_file_info.path
filename = os.path.basename(file_path).replace(".csv", "")
delta_table_name = f"{filename}_delta"

# Check if table already exists
tables = [t.name for t in spark.catalog.listTables()]
if delta_table_name in tables:
    print(f"Table '{delta_table_name}' already exists. Skipping creation.")
else:
    print(f"Creating new table '{delta_table_name}' from '{filename}.csv'...")

    # ====================================================
    # Step 1: Read CSV with first row as header
    # ====================================================
    df_temp = spark.read.csv(file_path, header=False, inferSchema=False)
    header_row = df_temp.first()
    df = df_temp.toDF(*[str(c) for c in header_row]).filter(F.col(header_row[0]) != header_row[0])

    # Clean column names
    def clean_column_name(c):
        return re.sub(r"[^0-9a-zA-Z_]", "_", c.strip())
    df = df.toDF(*[clean_column_name(c) for c in df.columns])

    # ====================================================
    # Step 2: Detect column types (sampled)
    # ====================================================
    sample_df = df.limit(500)
    cast_exprs = []

    for col_name in sample_df.columns:
        sample_vals = sample_df.select(col_name).na.drop()

        # ISO 8601 Timestamp
        if sample_vals.filter(F.col(col_name).rlike(r"^\d{4}[-/]\d{2}[-/]\d{2}T")).count() > 0:
            cast_exprs.append(F.to_timestamp(F.col(col_name), "yyyy/MM/dd'T'HH:mm:ss.SSS'Z'").alias(col_name))
            continue

        # Date formats
        if sample_vals.filter(F.col(col_name).rlike(r"^\d{4}-\d{2}-\d{2}$|^\d{2}-\d{2}-\d{4}$")).count() > 0:
            cast_exprs.append(F.to_date(F.col(col_name), "yyyy-MM-dd").alias(col_name))
            continue

        # Integer
        if sample_vals.filter(~F.col(col_name).rlike(r"^-?\d+$")).count() == 0:
            cast_exprs.append(F.col(col_name).cast(IntegerType()).alias(col_name))
            continue

        # Decimal
        if sample_vals.filter(~F.col(col_name).rlike(r"^-?\d+(\.\d+)?$")).count() == 0:
            cast_exprs.append(F.col(col_name).cast(DoubleType()).alias(col_name))
            continue

        # Default: String
        cast_exprs.append(F.col(col_name).cast(StringType()).alias(col_name))

    df_casted = df.select(*cast_exprs)

    # ====================================================
    # Step 3: Save as Delta
    # ====================================================
    df_casted.write.format("delta").mode("overwrite").saveAsTable(delta_table_name)
    print(f"Delta table '{delta_table_name}' created successfully with correct data types.")


In [0]:
%sql
 select * from default.TradeManager24_06_2025_delta


In [0]:
import os
from pyspark.sql import functions as F

base_path = "dbfs:/FileStore/TreadingSample/"

# Step 1: Get all CSV file names in base path
files = [f.name for f in dbutils.fs.ls(base_path) if f.name.endswith(".csv")]
# table_names = [os.path.splitext(f)[0] + "_delta" for f in files]
table_names = [t.name for t in spark.catalog.listTables("test_cfa")]
# Step 2: List existing tables from "default" schema
# existing_tables = [t.name for t in spark.catalog.listTables("default")]

print(table_names)

In [0]:
import os
from pyspark.sql import functions as F

base_path = "dbfs:/FileStore/TreadingSample/"

# Step 1: Get all CSV file names in base path
files = [f.name for f in dbutils.fs.ls(base_path) if f.name.endswith(".csv")]
# table_names = [os.path.splitext(f)[0] + "_delta" for f in files]
table_names = [t.name for t in spark.catalog.listTables("test_cfa")]

# Step 2: List existing tables from "default" schema
# existing_tables = [t.name for t in spark.catalog.listTables("default")]

all_stats = []

# Step 3: Loop through each table and collect stats
for tbl in table_names:
    # if tbl not in existing_tables:
    #     print(f"Skipping table '{tbl}' - not found in default schema.")
    #     continue

    # Read table from default schema
    df = spark.table(f"default.{tbl}")
    schema_info = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]

    # One-pass aggregation expressions
    agg_exprs = []
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            agg_exprs.extend([
                F.min(F.col(col_name)).cast("string").alias(f"{col_name}_min"),
                F.max(F.col(col_name)).cast("string").alias(f"{col_name}_max"),
                F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls")
            ])
        else:
            agg_exprs.append(F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls"))

    stats_row = df.agg(*agg_exprs).collect()[0]

    # Build final results list
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            all_stats.append((
                stats_row[f"{col_name}_min"],
                stats_row[f"{col_name}_max"],
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,
                tbl.replace("_delta", ".csv")
            ))
        else:
            all_stats.append((
                None,
                None,
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,
                tbl.replace("_delta", ".csv")
            ))

# Step 4: Create final DataFrame
final_stats_df = spark.createDataFrame(all_stats, ["min_value", "max_value", "null_count", "column_name", "data_type", "filename"])
display(final_stats_df)


In [0]:
%sql
select * from default.trademanager24_06_2025_delta

In [0]:
from pyspark.sql import functions as F

schema_name = "test_cfa"  # Change to your schema name

# Step 1: Get all tables in the schema
table_names = [t.name for t in spark.catalog.listTables(schema_name)]

all_stats = []

# Step 2: Loop through each table in schema
for tbl in table_names:
    print(f"Processing table: {schema_name}.{tbl}")
    df = spark.table(f"{schema_name}.{tbl}")
    schema_info = [(f.name, f.dataType.simpleString()) for f in df.schema.fields]

    # Build aggregation expressions for one-pass computation
    agg_exprs = []
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            agg_exprs.extend([
                F.min(F.col(col_name)).cast("string").alias(f"{col_name}_min"),
                F.max(F.col(col_name)).cast("string").alias(f"{col_name}_max"),
                F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls")
            ])
        else:
            agg_exprs.append(F.count(F.when(F.col(col_name).isNull(), 1)).alias(f"{col_name}_nulls"))

    # Aggregate
    stats_row = df.agg(*agg_exprs).collect()[0]

    # Build results list
    for col_name, dtype in schema_info:
        if dtype in ["int", "bigint", "double", "float", "decimal", "date", "timestamp"]:
            all_stats.append((
                stats_row[f"{col_name}_min"],
                stats_row[f"{col_name}_max"],
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,
                tbl.replace("_delta", ".csv")
            ))
        else:
            all_stats.append((
                None,
                None,
                stats_row[f"{col_name}_nulls"],
                col_name,
                dtype,
                tbl.replace("_delta", ".csv")
            ))

# Step 3: Create final DataFrame
final_stats_df = spark.createDataFrame(all_stats, ["min_value", "max_value", "null_count", "column_name", "data_type", "filename"])
display(final_stats_df)
